In [320]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import holidays
import warnings
import requests
warnings.filterwarnings('ignore')

In [321]:
train = pd.read_csv('labels/labels_day_train.csv', sep=';')
test = pd.read_csv('labels/labels_day_test.csv', sep=';')

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
display(train.head())

Train shape: (46234, 4), Test shape: (11317, 4)


,route,date,hour,boardings
0,1,2025-01-01,5,2
1,1,2025-01-01,6,26
2,1,2025-01-01,7,26
3,1,2025-01-01,8,26
4,1,2025-01-01,9,53


In [322]:
full_df = pd.concat([train, test], ignore_index=True)
full_df['date'] = pd.to_datetime(full_df['date'])
full_df = full_df.sort_values(by=['route', 'date', 'hour']).reset_index(drop=True)
print(f"Общий размер данных: {full_df.shape}")

Общий размер данных: (57551, 4)


In [323]:
def create_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

    ru_holidays = holidays.RU(years=[2025])
    df['is_holiday'] = df['date'].dt.date.apply(lambda x: x in ru_holidays).astype(int)
    
    #https://www.consultant.ru/law/ref/calendar/proizvodstvennye/
    major_events = ['2025-01-01','2025-01-02','2025-01-07','2025-02-23','2025-03-08',
                    '2025-05-01','2025-05-09','2025-06-12','2025-09-13','2025-11-04','2025-12-31']
    df['is_major_event'] = df['date'].dt.strftime('%Y-%m-%d').isin(major_events).astype(int)

    #https://www.rbc.ru/life/news/650956c49a7947086a7f6d64 
    holiday_str = (
        # Модульная система (RBC)
        list(pd.date_range('2025-10-04', '2025-10-12').strftime('%Y-%m-%d')) +
        list(pd.date_range('2025-11-15', '2025-11-23').strftime('%Y-%m-%d')) +
        # Четвертная система (Минпросвещения РФ)
        list(pd.date_range('2025-10-25', '2025-11-02').strftime('%Y-%m-%d')) +
        # Зимние (общие для обеих систем)
        list(pd.date_range('2025-12-31', '2025-12-31').strftime('%Y-%m-%d'))
    )
    df['is_school_break'] = df['date'].dt.strftime('%Y-%m-%d').isin(holiday_str).astype(int)

    #среднее от https://data.mos.ru/opendata/62525
    def get_traffic_index(row):
        h = row['hour']
        is_we = row['is_weekend'] or row['is_holiday']
        if is_we:
            return 4.0 if 12 <= h <= 19 else 2.0  # Суббота/Воскресенье без пиков
        else: # Будни
            if 0 <= h <= 6:   return 1.0          # Ночной межпик (LOS A)
            if 7 <= h <= 11:  return 8.0          # Утренний пик (LOS E)
            if 12 <= h <= 15: return 5.0          # Дневной межпик (LOS C)
            if h == 16:       return 5.0          # Переходный час
            if 17 <= h <= 20: return 8.0          # Вечерний пик (LOS E)
            if h == 21:       return 4.0          # Переходный час
            if 22 <= h <= 23: return 2.0          # Ночной межпик (LOS A/B)
            return 1.0

    df['traffic_index'] = df.apply(get_traffic_index, axis=1)
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    df['route'] = df['route'].astype(int)
    df['hour'] = df['hour'].astype(int)
    
    if 'boardings' in df.columns:
        df['route_mean_boardings'] = df.groupby('route')['boardings'].transform('mean')
        df['route_hour_mean'] = df.groupby(['route', 'hour'])['boardings'].transform('mean')
    else:
        df['route_mean_boardings'] = 0
        df['route_hour_mean'] = 0
    
    return df

full_df = create_features(full_df)
print("Признаки созданы")


Признаки созданы


In [324]:
# url = "https://archive-api.open-meteo.com/v1/archive"
# params = {
#     'latitude': 55.7558,
#     'longitude': 37.6173,
#     'start_date': '1995-11-01',
#     'end_date': '2024-12-31',
#     'daily': 'temperature_2m_mean,precipitation_sum',
#     'timezone': 'Europe/Moscow'
# }

# print("Запрос исторических данных из Open-Meteo API (ERA5)...")
# r = requests.get(url, params=params, timeout=30)
# data = r.json()

# if 'daily' not in data:
#     print(f"API вернул ошибку: {data}")
#     print("Используем кэшированные значения (fallback в add_weather_features).")
# else:
#     df_api = pd.DataFrame({
#         'date': pd.to_datetime(data['daily']['time']),
#         'temp': data['daily']['temperature_2m_mean'],
#         'precip': data['daily']['precipitation_sum']
#     })
#     df_api['month'] = df_api['date'].dt.month
#     api_climate = df_api.groupby('month').agg({'temp': 'mean', 'precip': 'mean'}).round(1)
#     print("Проверенные климатические нормы:")
#     print(api_climate)

In [325]:
def add_weather_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    month_series = df['date'].dt.month
    
    temp_map = {1:-7.7, 2:-7.3, 3:-2.5, 4:5.7, 5:12.9, 6:17.4, 
                7:19.9, 8:18.1, 9:12.2, 10:5.8, 11:-0.5, 12:-5.2}
    
    precip_map = {1:34.7, 2:32.7, 3:32.5, 4:38.3, 5:57.4, 6:64.7, 
                  7:74.2, 8:62.9, 9:52.4, 10:55.6, 11:48.2, 12:42.8}

    df['temp'] = month_series.map(temp_map)
    df['precipitation'] = month_series.map(precip_map)
    
    df['is_cold'] = (df['temp'] <= -5).astype(int)
    df['is_snow'] = ((df['precipitation'] > 0) & (df['temp'] < 0)).astype(int)
    
    return df

full_df = add_weather_features(full_df)


In [326]:
train_mask = full_df['date'] < '2025-09-01'
val_mask = (full_df['date'] >= '2025-09-01') & (full_df['date'] <= '2025-10-31')

features = ['route', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_holiday', 
            'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
            'temp', 'precipitation',
            'route_mean_boardings', 'route_hour_mean',
            'is_cold', 'is_snow',
            'is_major_event', 'traffic_index',
            'is_school_break']

target = 'boardings'

X_train = full_df[train_mask][features]
y_train = full_df[train_mask][target]
X_val = full_df[val_mask][features]
y_val = full_df[val_mask][target]

print(f"Размер Train: {X_train.shape}, Размер Validation: {X_val.shape}")

Размер Train: (46234, 19), Размер Validation: (11317, 19)


In [327]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

In [328]:
params = {
    'objective': 'poisson',
    'metric': 'mae',
    'learning_rate': 0.005,
    'num_leaves': 95,
    'max_depth': 10,
    'feature_fraction': 0.2,
    'bagging_fraction': 0.92,
    'bagging_freq': 1,
    'min_data_in_leaf': 30,
    'verbose': -1,
    'random_state': 42
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=22000,
    callbacks=[lgb.early_stopping(300), lgb.log_evaluation(1000)]
)

Training until validation scores don't improve for 300 rounds
[1000]	valid_0's l1: 274.046
[2000]	valid_0's l1: 164.748
[3000]	valid_0's l1: 143.911
[4000]	valid_0's l1: 138.897
[5000]	valid_0's l1: 137.467
[6000]	valid_0's l1: 137.167
Early stopping, best iteration is:
[6412]	valid_0's l1: 137.091


In [329]:
val_preds = model.predict(X_val)
val_preds = np.maximum(0, val_preds)

def wape(y_true, y_pred):
    return 1 - np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

wape_score = wape(y_val.values, val_preds)
print(f"WAPE на валидации (Сен-Окт): {wape_score:.4f}")
print(f"Baseline для сравнения: 0.48")

WAPE на валидации (Сен-Окт): 0.8784
Baseline для сравнения: 0.48


In [330]:
bias_factor = np.sum(y_val.values) / np.sum(val_preds)
print(f"Коэффициент смещения: {bias_factor:.4f}")

val_preds_corrected = val_preds * bias_factor

wape_corrected = wape(y_val.values, val_preds_corrected)
print(f"WAPE после коррекции: {wape_corrected:.4f}")

Коэффициент смещения: 1.0288
WAPE после коррекции: 0.8815


In [331]:
print("Генерация прогноза на Ноябрь-Декабрь")

routes = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]
dates = pd.date_range(start='2025-11-01', end='2025-12-31', freq='D')
hours = list(range(24))

grid = pd.MultiIndex.from_product([routes, dates, hours], names=['route','date','hour']).to_frame(index=False)
grid = create_features(grid)
grid = add_weather_features(grid)

route_means = full_df.groupby('route')['boardings'].mean().to_dict()
grid['route_mean_boardings'] = grid['route'].map(route_means).fillna(0)

route_hour_means = full_df.groupby(['route','hour'])['boardings'].mean().to_dict()
grid['route_hour_mean'] = grid.set_index(['route','hour']).index.map(route_hour_means).fillna(0)

print(f"Сетка: {grid.shape}")

Генерация прогноза на Ноябрь-Декабрь
Сетка: (14640, 20)


In [332]:
print("Генерация прогноза (Bias: 7 дней недели × Часы)")

val_preds_raw = model.predict(X_val)
val_preds_raw = np.maximum(0, val_preds_raw)

val_analysis = X_val.copy().reset_index(drop=True)
val_analysis['actual'] = y_val.values
val_analysis['preds'] = val_preds_raw
val_analysis['hour_int'] = val_analysis['hour'].astype(int)
val_analysis['dow'] = val_analysis['dayofweek'].astype(int)

def get_bias_subset(df, h_start, h_end):
    subset = df[(df['hour_int'] >= h_start) & (df['hour_int'] <= h_end)]
    if subset['preds'].sum() == 0: return 1.0
    return subset['actual'].sum() / subset['preds'].sum()

bias_dict = {}
for d in range(7):
    grp_df = val_analysis[val_analysis['dow'] == d]
    if len(grp_df) == 0:
        continue
    bias_dict[d] = {
        'night':   get_bias_subset(grp_df, 0, 6),
        'morning': get_bias_subset(grp_df, 7, 11),
        'day':     get_bias_subset(grp_df, 12, 16),
        'evening': get_bias_subset(grp_df, 17, 23)
    }
    print(f"Day {d}: N={bias_dict[d]['night']:.3f}, M={bias_dict[d]['morning']:.3f}, "
          f"D={bias_dict[d]['day']:.3f}, E={bias_dict[d]['evening']:.3f}")

grid_preds = model.predict(grid[features])
grid['prediction'] = np.maximum(0, grid_preds)
grid['hour_int'] = grid['hour'].astype(int)
grid['dow'] = grid['dayofweek'].astype(int)

for d in range(7):
    if d not in bias_dict:
        continue
    m = grid['dow'] == d
    grid.loc[m & (grid['hour_int'] <= 6), 'prediction'] *= bias_dict[d]['night']
    grid.loc[m & (grid['hour_int'] >= 7) & (grid['hour_int'] <= 11), 'prediction'] *= bias_dict[d]['morning']
    grid.loc[m & (grid['hour_int'] >= 12) & (grid['hour_int'] <= 16), 'prediction'] *= bias_dict[d]['day']
    grid.loc[m & (grid['hour_int'] >= 17), 'prediction'] *= bias_dict[d]['evening']

grid['prediction'] = grid['prediction'].round().astype(int)
grid.loc[grid['route'] == 5, 'prediction'] = 0

submission = grid[['route', 'date', 'hour', 'prediction']].copy()
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission = submission.drop_duplicates(subset=['route', 'date', 'hour'])
submission.to_csv('submission.csv', sep=';', index=False)
print(f"Сабмит сохранен! Строк: {len(submission)}")
display(submission.head())

Генерация прогноза (Bias: 7 дней недели × Часы)
Day 0: N=0.956, M=1.052, D=1.070, E=0.997
Day 1: N=0.960, M=1.056, D=1.051, E=0.996
Day 2: N=0.959, M=1.036, D=1.039, E=0.994
Day 3: N=0.953, M=1.037, D=1.052, E=0.978
Day 4: N=0.979, M=1.051, D=1.035, E=0.985
Day 5: N=0.990, M=1.074, D=1.081, E=0.986
Day 6: N=0.966, M=1.050, D=1.103, E=1.000
Сабмит сохранен! Строк: 14640


,route,date,hour,prediction
0,1,2025-11-01,0,15
1,1,2025-11-01,1,6
2,1,2025-11-01,2,25
3,1,2025-11-01,3,40
4,1,2025-11-01,4,10


In [333]:
import os, json, pickle, shutil
import pandas as pd

os.makedirs('artifacts', exist_ok=True)

assert model.params.get('objective') == 'poisson', "Не Poisson!"
with open('artifacts/model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("model.pkl")

route_hour_means_str = {f"{r}|{h}": float(v) for (r, h), v in route_hour_means.items()}
bias_dict_str = {str(k): {kk: float(vv) for kk, vv in v.items()} for k, v in bias_dict.items()}

metadata = {
    'model_type': 'LightGBM Booster',
    'objective': 'poisson',
    'target_transform': None,
    'features_order': features,
    'n_features': len(features),
    'routes': routes,
    'hours': hours,
    'route_means': {str(k): float(v) for k, v in route_means.items()},
    'route_hour_means': route_hour_means_str,
    'bias_dict': bias_dict_str,
    'temp_map': {1:-7.7, 2:-7.3, 3:-2.5, 4:5.7, 5:12.9, 6:17.4, 7:19.9, 8:18.1, 9:12.2, 10:5.8, 11:-0.5, 12:-5.2},
    'precip_map': {1:34.7, 2:32.7, 3:32.5, 4:38.3, 5:57.4, 6:64.7, 7:74.2, 8:62.9, 9:52.4, 10:55.6, 11:48.2, 12:42.8},

    'traffic_monthly_map': {
        '9': 2.1,   # Сентябрь 2025
        '10': 2.2,  # Октябрь 2025
        '11': 2.8,  # Ноябрь 2025 (Период теста)
        '12': 2.8   # Декабрь 2025 (Период теста)
    },
    # https://www.garant.ru/products/ipo/prime/doc/406101059/#review Распоряжение Министерства транспорта РФ от 27 декабря 2022 г.
    # № АК-337-р “Об утверждении методических рекомендаций по проведению мониторинга дорожного движения”
    'traffic_index_logic': 'weekday: 0-6=1, 7-11=8, 12-16=5, 17-20=8, 21=4, 22-23=2; weekend: 12-19=4, else=2',
    'valid_period': {'from': '2025-11-01', 'to': '2025-12-31'},
    'wape_val': float(wape_score),
    'wape_corrected': float(wape_corrected),
    'bias_hour_blocks': {'night': [0,6], 'morning': [7,11], 'day': [12,16], 'evening': [17,23]},
    'school_breaks': {
        'name': 'Каникулы 2025/2026 (модульная + четвертная системы)',
        'url': 'https://www.rbc.ru/life/news/650956c49a7947086a7f6d64',
        'features': ['is_school_break']
    },
    'external_sources': {
        'weather': {'name': 'Open-Meteo Archive API (ERA5)', 'url': 'https://archive-api.open-meteo.com/v1/archive'},
        'calendar': {'name': 'Производственный календарь РФ', 'url': 'https://www.consultant.ru/law/ref/calendar/proizvodstvennye/'},
        'traffic': {'name': 'ЦОДД / эвристика часа пик', 'url': 'https://data.mos.ru/opendata/62525'},
        'events': {'name': 'Афиша Москвы', 'url': 'https://www.mos.ru/afisha/'}
    }
}

with open('artifacts/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("metadata.json")

shutil.copy('submission.csv', 'artifacts/submission.csv')
print("submission.csv")

sub = pd.read_csv('artifacts/submission.csv', sep=';')
v = sub[(sub['route']==1) & (sub['date']=='2025-11-01') & (sub['hour']==8)]['prediction'].values[0]
print(f"\nCSV route=1, 2025-11-01, hour=8: {v}")
print(f"bias_dict[5]['morning'] в metadata.json: {bias_dict_str['5']['morning']:.4f}")
print(f"route_hour_means['1|8'] в metadata.json: {route_hour_means_str['1|8']:.2f}")

model.pkl
metadata.json
submission.csv

CSV route=1, 2025-11-01, hour=8: 507
bias_dict[5]['morning'] в metadata.json: 1.0744
route_hour_means['1|8'] в metadata.json: 1332.51
